<a href="https://colab.research.google.com/github/souro26/bayesian-a-b-testing/blob/main/examples/05_personalisation_segments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import argonx  # noqa: F401
except ImportError:
    !pip install -q git+https://github.com/souro26/bayesian-a-b-testing.git

# Example 05 — Personalisation Strategy & Hierarchical Pooling

A fintech app is testing a 'Smart Savings' feature. The goal is to increase the amount of money users transfer into their savings accounts.

### The Problem: Segment Contradictions
In a global A/B test, a feature might look like a 'neutral' result, but that neutral could be hiding a **major win** in one segment (e.g., iOS users) and a **major loss** in another (e.g., Android users). 

### The Solution: Hierarchical Modeling with Partial Pooling
Hierarchical models allow us to model each segment individually while still acknowledging they belong to the same global population. 

**Partial Pooling** is the magic here: 
1.  Segments with lots of data (like Mobile) are allowed to 'speak for themselves'.
2.  Segments with tiny amounts of data (like Tablet) are 'shrunk' toward the global average. This prevents us from making wild decisions based on the noise of a few users, while still giving us a better estimate than simply ignoring the segment.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from argonx import Experiment

plt.style.use("seaborn-v0_8-muted")
sns.set_theme(style="whitegrid")

## 1. Simulating Hierarchical Data

We'll simulate three platforms:
*   **iOS:** High traffic, clear positive lift (+20%).
*   **Android:** High traffic, neutral/slightly negative signal.
*   **Tablet:** Extremely low traffic (20 users!), very noisy.

In [ ]:
np.random.seed(99)


def gen_rev(n, median, sigma, lift):
    c = np.random.lognormal(np.log(median), sigma, n)
    v = np.random.lognormal(np.log(median * lift), sigma, n)
    return c, v


# iOS: Clear win
c_ios, v_ios = gen_rev(1500, 100, 0.4, 1.20)

# Android: Neutral
c_and, v_and = gen_rev(1500, 85, 0.4, 1.0)

# Tablet: Tiny data (20 users), looks like 50% lift by pure luck!
c_tab, v_tab = gen_rev(20, 90, 0.4, 1.50)

df = pd.DataFrame(
    {
        "platform": (["iOS"] * 3000 + ["Android"] * 3000 + ["Tablet"] * 40),
        "variant": (
            ["control"] * 1500
            + ["variant_b"] * 1500
            + ["control"] * 1500
            + ["variant_b"] * 1500
            + ["control"] * 20
            + ["variant_b"] * 20
        ),
        "deposit_amount": np.concatenate([c_ios, v_ios, c_and, v_and, c_tab, v_tab]),
    }
)

print("Observed Medians per Segment:")
print(df.groupby(["platform", "variant"])["deposit_amount"].median().unstack())

## 2. Running the Hierarchical Experiment

We define `segment_col='platform'`. `argonx` will automatically fit a hierarchical model using MCMC, sharing information across platforms to stabilize the estimates.

In [ ]:
exp = Experiment(
    data=df,
    variant_col="variant",
    segment_col="platform",
    primary_metric="deposit_amount",
    model="lognormal",
    control="control",
)

result = exp.run()
result.summary()

## 3. The Population vs. Segment Dashboard

First, we see the aggregate view. It looks like a moderate win.

In [ ]:
result.plot(suptitle="Global Population Result")

Now we use `plot_segments()` to see the truth. Notice how **Tablet** is handled: even though the raw data showed a 50% lift, the model correctly stays cautious because the sample size is tiny.

In [ ]:
result.plot_segments()

### Conclusion

This hierarchical analysis saved us from two potential mistakes:
1.  **Missing the iOS Win:** In a flat analysis, the Android neutral result might have diluted the iOS signal.
2.  **Chasing the Tablet Noise:** The Raw Tablet data looked incredible (50% lift), but the hierarchical model correctly 'shrunk' that estimate back toward reality because 20 users isn't enough to bank on.

**Strategy Recommendation:** Roll out the feature to **iOS users immediately**, investigate the UX friction on **Android**, and keep monitoring **Tablet** as more data arrives.